In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from backtesting import Backtest, Strategy 
from backtesting.lib import crossover, SignalStrategy, TrailingStrategy

from backtesting.test import SMA, GOOG

/opt/anaconda3/lib/python3.13/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

Приведем пример стратегии торговли только на длинных позициях со стоплоссом.



In [2]:
class SmaCross(Strategy):
    # Определите два запаздывания MA как **переменные класса**
    # для последующей оптимизации
    n1 = 10
    n2 = 30

    def init(self):
        # Предварительно вычислим две скользящие средние
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
        
    def next(self):
        price = self.data.Close[-1]
        
        # Если sma1 пересекается выше sma2, закройте все существующие короткие позиции
        # и купите актив
        # важно стоплосс настраивается по сигналу на покупку акций, а не по цене покупки (как перестроить пока не знаю)

        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy(sl = price*0.95)

        # В противном случае, если sma1 пересечется ниже sma2, закройте все существующие 
        # длинные сделки и продайте актив
        elif crossover(self.sma2, self.sma1):
            self.position.close()
               

In [10]:
bt = Backtest (GOOG, SmaCross, cash=10_000, commission=0, finalize_trades=True)

stats=bt.run()
bt.plot()

Backtest.run:   0%|          | 0/2118 [00:00<?, ?bar/s]

GridPlot(id='p3603', ...)

In [8]:
stats['_trades']  # Contains individual trade data

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,"Entry_SMA(C,10)","Exit_SMA(C,10)","Entry_SMA(C,30)","Exit_SMA(C,30)"
0,53,86,113,186.31,193.6900,175.7690,None,391.1400,0.0,0.039611,2004-12-21,2005-01-31,41 days,None,176.930,190.452,175.833667,191.597667
1,52,119,121,196.96,186.2285,186.2285,None,-558.0380,0.0,-0.054486,2005-02-08,2005-02-10,2 days,None,197.103,197.327,194.690667,194.487333
2,50,160,166,193.69,184.0720,184.0720,None,-480.9000,0.0,-0.049657,2005-04-08,2005-04-18,10 days,None,185.088,190.714,182.942333,183.793333
3,31,267,294,297.28,304.9600,280.6205,None,238.0800,0.0,0.025834,2005-09-09,2005-10-18,39 days,None,289.646,305.603,287.938667,307.097667
4,27,299,365,345.78,430.5700,331.2175,None,2289.3300,0.0,0.245214,2005-10-25,2006-01-31,98 days,None,315.019,431.159,311.650667,439.145667
5,30,408,434,389.53,408.3100,370.5000,None,563.4000,0.0,0.048212,2006-04-03,2006-05-10,37 days,None,369.784,402.145,362.807333,407.289333
6,32,459,486,386.62,385.0200,365.1705,None,-51.2000,0.0,-0.004138,2006-06-15,2006-07-25,40 days,None,385.370,399.701,383.065000,403.757333
7,32,518,585,376.72,484.6900,359.5655,None,3455.0400,0.0,0.286605,2006-09-08,2006-12-13,96 days,None,379.188,483.762,378.240000,485.036333
8,31,604,619,501.99,474.7340,474.7340,None,-844.9360,0.0,-0.054296,2007-01-12,2007-02-05,24 days,None,482.434,488.075,478.023333,482.177000
9,32,657,686,462.10,461.8300,437.8740,None,-8.6400,0.0,-0.000584,2007-03-30,2007-05-11,42 days,None,458.251,468.205,456.386000,470.626333
